<a href="https://colab.research.google.com/github/VidiDamar/VidiDamar/blob/main/Uprak_ML_TeachableMachine_DecisionEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Touchless Order — Asisten Layanan / Kantin Tanpa Sentuh
**XII RPL 2 | Machine Learning**

In [1]:
nama_kelompok = "Kelompok 5"
kelas = "XII RPL 2"
anggota = [
    "Abdurrahman Arief - Data Collector",
    "Abdurrahman Arief dan Fathir - Model Trainer",
    "Fathir- Python Programmer",
    "Hans- QA Analyst / Video Editor",
]
link_teachable_machine = https://colab.research.google.com/drive/1tKEBoto5V5HIzt2HMpcStFBmWYXTAjto?usp=sharing

CONFIDENCE_THRESHOLD = 75.0

sampel_uji = [
    ("Pesanan 1 - Gestur/suara menu 1 (jarak normal)", "Paket Nasi Ayam", 94.0),
    ("Pesanan 2 - Gestur/suara menu 2 (cahaya cukup)", "Mie Goreng", 89.5),
    ("Pesanan 3 - Gestur/suara menu 3 (jelas)", "Es Teh Manis", 91.2),
    ("Kondisi 4 - Diam / suara latar belakang", "Netral/Noise", 95.0),
    ("Kondisi 5 - Gestur buram / suara pelan", "Paket Nasi Ayam", 60.2),
]

print(f"Kelompok   : {nama_kelompok} ({kelas})")
print(f"Studi Kasus: Kasus 5 - Touchless Order (Kantin Tanpa Sentuh)")
print(f"Link Model : {link_teachable_machine}")
print(f"Sampel Uji : {len(sampel_uji)} data (Threshold: {CONFIDENCE_THRESHOLD}%)")

SyntaxError: invalid decimal literal (3947376759.py, line 9)

### Uji Langsung Model Image Keras Teachable Machine (Opsional)
Khusus jika menggunakan **Image Project** dan ingin tes langsung dari file export `keras_model.h5` & `labels.txt`.

In [2]:
JALANKAN_MODEL_LANGSUNG = False

if JALANKAN_MODEL_LANGSUNG:
    import os
    os.environ["TF_USE_LEGACY_KERAS"] = "1"

    import numpy as np
    from PIL import Image, ImageOps
    from google.colab import files

    try:
        import tf_keras as keras
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tf_keras"])
        import tf_keras as keras

    print("Upload keras_model.h5 dan labels.txt:")
    files.upload()

    print("Upload gambar uji:")
    uploaded_images = files.upload()

    np.set_printoptions(suppress=True)
    model = keras.models.load_model("keras_model.h5", compile=False)
    with open("labels.txt", "r", encoding="utf-8") as f:
        class_names = [line.strip() for line in f.readlines() if line.strip()]

    def bersihkan_label(raw_label):
        bagian = raw_label.split(" ", 1)
        return bagian[1] if len(bagian) > 1 and bagian[0].isdigit() else raw_label

    hasil_uji_colab = []
    for nama_file in uploaded_images.keys():
        data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
        image = Image.open(nama_file).convert("RGB")
        image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)
        image_array = np.asarray(image)
        data[0] = (image_array.astype(np.float32) / 127.5) - 1

        prediction = model.predict(data, verbose=0)
        index = int(np.argmax(prediction))
        label_prediksi = bersihkan_label(class_names[index])
        confidence_score = round(float(prediction[0][index]) * 100, 2)

        print(f"{nama_file} -> {label_prediksi} ({confidence_score}%)")
        hasil_uji_colab.append((f"Upload Colab: {nama_file}", label_prediksi, confidence_score))

    if hasil_uji_colab:
        sampel_uji = hasil_uji_colab
else:
    print("Mode Preview aktif.")

Mode Preview aktif.


### Function (`def`) & Logika Keputusan Pesanan (`if - elif - else`)

In [3]:
def evaluasi_pesanan(label: str, confidence: float, threshold: float = 75.0) -> dict:
    if confidence < threshold:
        return {
            "diterima": False,
            "status": "DIABAIKAN (RAGU)",
            "menu": "-",
            "harga": 0,
            "keterangan": f"Confidence {confidence:.1f}% < {threshold:.0f}%, ulangi input / cek manual",
        }

    if label == "Netral/Noise":
        return {
            "diterima": False,
            "status": "DIABAIKAN (NETRAL)",
            "menu": "-",
            "harga": 0,
            "keterangan": "Input netral/noise diabaikan",
        }
    elif label == "Paket Nasi Ayam":
        return {
            "diterima": True,
            "status": "PESANAN VALID",
            "menu": "Paket Nasi Ayam",
            "harga": 18000,
            "keterangan": "Masuk daftar pesanan (Rp18.000)",
        }
    elif label == "Mie Goreng":
        return {
            "diterima": True,
            "status": "PESANAN VALID",
            "menu": "Mie Goreng",
            "harga": 12000,
            "keterangan": "Masuk daftar pesanan (Rp12.000)",
        }
    elif label == "Es Teh Manis":
        return {
            "diterima": True,
            "status": "PESANAN VALID",
            "menu": "Es Teh Manis",
            "harga": 5000,
            "keterangan": "Masuk daftar pesanan (Rp5.000)",
        }
    else:
        return {
            "diterima": False,
            "status": "TIDAK DIKENAL",
            "menu": label,
            "harga": 0,
            "keterangan": "Label tidak terdaftar di menu kantin",
        }


def hitung_total_bayar(daftar_hasil: list) -> dict:
    total_bayar = 0
    daftar_menu = []
    jumlah_diabaikan = 0

    for item in daftar_hasil:
        if item["diterima"]:
            daftar_menu.append((item["menu"], item["harga"]))
            total_bayar += item["harga"]
        else:
            jumlah_diabaikan += 1

    return {
        "daftar_menu": daftar_menu,
        "jumlah_item": len(daftar_menu),
        "jumlah_diabaikan": jumlah_diabaikan,
        "total_bayar": total_bayar,
    }

### Proses Daftar Pesanan (`for`) & Total Bayar

In [4]:
daftar_hasil = []

print("=" * 68)
print(f"SISTEM KANTIN TANPA SENTUH (TOUCHLESS ORDER) - {nama_kelompok}")
print("=" * 68)

for nomor, (kondisi, label, confidence) in enumerate(sampel_uji, start=1):
    hasil = evaluasi_pesanan(label, confidence, CONFIDENCE_THRESHOLD)
    daftar_hasil.append(hasil)

    print(f"[{nomor}] Kondisi    : {kondisi}")
    print(f"    Prediksi   : {label} ({confidence:.1f}%)")
    print(f"    Status     : {hasil['status']}")
    print(f"    Keterangan : {hasil['keterangan']}")
    print("-" * 68)

rekap = hitung_total_bayar(daftar_hasil)

print("DAFTAR PESANAN VALID:")
if rekap["daftar_menu"]:
    for idx, (menu, harga) in enumerate(rekap["daftar_menu"], start=1):
        print(f"  {idx}. {menu:<20} Rp{harga:,}")
else:
    print("  (Tidak ada pesanan valid)")

print("-" * 68)
print(f"Total Item Dipesan : {rekap['jumlah_item']}")
print(f"Input Diabaikan    : {rekap['jumlah_diabaikan']}")
print(f"TOTAL BAYAR        : Rp{rekap['total_bayar']:,}")
print("=" * 68)

NameError: name 'nama_kelompok' is not defined

### Uji Input Live Demo (`while`)

In [ ]:
JALANKAN_INTERAKTIF = False

if JALANKAN_INTERAKTIF:
    hasil_live = []
    print("Input Pesanan Live (ketik 'selesai' untuk menghitung total bayar):")
    while True:
        label_input = input("\nLabel Prediksi Teachable Machine: ").strip()
        if label_input.lower() == "selesai":
            break
        try:
            conf_input = float(input("Confidence Score (%): ").strip())
        except ValueError:
            print("Confidence harus berupa angka.")
            continue

        hasil_item = evaluasi_pesanan(label_input, conf_input, CONFIDENCE_THRESHOLD)
        hasil_live.append(hasil_item)
        print(f"-> {hasil_item['status']}: {hasil_item['keterangan']}")

    if hasil_live:
        rekap_live = hitung_total_bayar(hasil_live)
        print(f"\nTotal Item: {rekap_live['jumlah_item']} | Total Bayar: Rp{rekap_live['total_bayar']:,}")
else:
    print("Mode interaktif nonaktif.")